In [1]:
a = pd.read_csv('../data/gender/transactions.csv') 

descr_mcc = pd.read_csv('../data/gender/tr_mcc_codes.csv', sep=';')
descr_types = pd.read_csv('../data/gender/tr_types.csv', sep=';')

a = a.merge(descr_mcc, on='mcc_code', how='left').merge(descr_types, on='tr_type', how='left')\
        .fillna({'tr_description' : "", 'mcc_description' : ""})

a['description'] = a.mcc_description + " " +  a.tr_description

a.drop(columns = ['mcc_description', 'tr_description']).to_csv('../data/gender/transactions_d.csv')

NameError: name 'pd' is not defined

In [2]:
import argparse
import datetime
import logging
import os
import pickle
from random import Random
import yaml
from tqdm.notebook import tqdm

import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt

from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer, util
import fasttext

from ptls.preprocessing.util import pd_hist
from dotsi import Dict
import re

logger = logging.getLogger(__name__)

In [2]:
trans = pd.read_csv(os.path.join('../data/gender', 'transactions_d.csv'))
targets = pd.read_csv(os.path.join('../data/gender', 'gender_train.csv'))

In [3]:
padded_time = trans['tr_datetime'].str.pad(15, 'left', '0')
day_part = padded_time.str[:6].astype(float)
time_part = pd.to_datetime(padded_time.str[7:], format='%H:%M:%S').values.astype('int64') // 1e9
time_part = time_part % (24 * 60 * 60) / (24 * 60 * 60)

In [4]:
trans['tr_datetime1'] = day_part + time_part

In [5]:
trans = trans.sort_values(by=['customer_id', 'tr_datetime1'])
trans['range_id'] = trans.groupby('customer_id').cumcount() // 300

trans['customer_id_new'] = trans['customer_id'].astype(str) + '_' + trans['range_id'].astype(str)

In [6]:
trans = trans.merge(pd.DataFrame({'customer_id_new' : trans['customer_id_new'].unique(), 'customer_id_new_n' : np.arange(trans['customer_id_new'].unique().shape[0])}), 
               on='customer_id_new', how='inner').drop(columns=['customer_id_new']).rename(columns={'customer_id_new_n' : 'customer_id_new'})

In [7]:
trans

,customer_id,tr_datetime,mcc_code,tr_type,amount,term_id,description,tr_datetime1,range_id,customer_id_new
0,6815,10 10:52:00,4814,1030,-2245.92,NaN,"Звонки с использованием телефонов, считывающих...",10.452778,0,0
1,6815,10 14:44:36,4814,1030,-2245.92,NaN,"Звонки с использованием телефонов, считывающих...",10.614306,0,0
2,6815,11 10:23:14,6010,7031,2470507.35,NaN,Финансовые институты — снятие наличности вручн...,11.432801,0,0
3,6815,14 12:49:46,6011,2010,-11229.58,NaN,Финансовые институты — снятие наличности автом...,14.534560,0,0
4,6815,17 12:33:31,4814,1030,-2245.92,NaN,"Звонки с использованием телефонов, считывающих...",17.523275,0,0
...,...,...,...,...,...,...,...,...,...,...
6849341,99999680,444 00:00:00,5411,1110,-5659.71,31190432,"Бакалейные магазины, супермаркеты Покупка. POS...",444.000000,1,30796
6849342,99999680,444 00:00:00,6011,2110,-134754.95,406826,Финансовые институты — снятие наличности автом...,444.000000,1,30796
6849343,99999680,446 00:00:00,5541,1110,-11229.58,J038003,Станции техобслуживания Покупка. POS ТУ Россия,446.000000,1,30796
6849344,99999680,451 09:56:17,6010,7070,1122.96,945022,Финансовые институты — снятие наличности вручн...,451.414086,1,30796


In [8]:
targets = targets.merge(trans[['customer_id_new', 'customer_id']].drop_duplicates(), on='customer_id', how='inner')

In [9]:
targets = targets.drop(columns=['customer_id']).rename(columns={'customer_id_new':'customer_id'})

In [10]:
targets

,Unnamed: 0,gender,customer_id
0,0,1,3343
1,0,1,3344
2,0,1,3345
3,1,1,21088
4,1,1,21089
...,...,...,...
16940,8396,0,20290
16941,8396,0,20291
16942,8397,1,3287
16943,8398,0,3475


In [11]:
trans = trans.drop(columns=['customer_id', 'tr_datetime1', 'range_id']).rename(columns={'customer_id_new':'customer_id'})
trans = trans.merge(targets[['customer_id']].drop_duplicates(), on='customer_id', how='inner')

In [14]:
trans.columns[1:-1]

Index(['mcc_code', 'tr_type', 'amount', 'term_id', 'description'], dtype='object')

In [16]:
trans[['customer_id'] + list(trans.columns[:-1])].to_csv('../data/gender/transactions_s_d.csv')

In [17]:
targets[['customer_id', 'gender']].to_csv('../data/gender/gender_train_s_d.csv')

In [18]:
trans_d = pd.read_csv(os.path.join('../data/gender', 'transactions_s_d.csv'))
trans = pd.read_csv(os.path.join('../data/gender', 'transactions_s.csv'))

In [ ]:
#age supervised trans cat

In [131]:
data_path = '../data/age_bins'

trans = pd.read_csv(os.path.join(data_path, 'transactions_train.csv'))
targets = pd.read_csv(os.path.join('../data/age_bins', 'train_target.csv')) 

In [132]:
trans = trans.sort_values(by=['client_id', 'trans_date'])
trans['range_id'] = trans.groupby('client_id').cumcount() // 300

trans['client_id_new'] = trans['client_id'].astype(str) + '_' + trans['range_id'].astype(str)

In [134]:
trans = trans.merge(pd.DataFrame({'client_id_new' : trans['client_id_new'].unique(), 'client_id_new_n' : np.arange(trans['client_id_new'].unique().shape[0])}), 
               on='client_id_new', how='inner').drop(columns=['client_id_new']).rename(columns={'client_id_new_n' : 'client_id_new'})

In [135]:
targets = targets.merge(trans[['client_id_new', 'client_id']].drop_duplicates(), on='client_id', how='inner')
targets = targets.drop(columns=['client_id']).rename(columns={'client_id_new':'client_id'})


In [139]:
trans = trans.drop(columns=['client_id', 'range_id']).rename(columns={'client_id_new':'client_id'})
trans = trans.merge(targets[['client_id']].drop_duplicates(), on='client_id', how='inner')

In [144]:
trans[['client_id'] + list(trans.columns[:-1])].to_csv('../data/age_bins/transactions_train_s.csv')
targets[['client_id', 'bins']].to_csv('../data/age_bins/train_target_s.csv')

In [ ]:
#x5 supervised trans cat

In [31]:
data_path = '../data/x5'

trans = pd.read_parquet(os.path.join(data_path, 'features_sample.parquet'))
targets = pd.read_parquet(os.path.join(data_path, 'targets_sample.parquet')) 

In [ ]:
trans

In [36]:
trans = trans.merge((trans.groupby('client_id')[['ordercol']].count() <= 375).reset_index().rename(columns={'ordercol':'keep'}).sample(40000, random_state=42),
                    on='client_id', how='inner')

In [38]:
trans = trans[trans.keep == True].drop(columns=['keep'])

In [39]:
trans[['client_id']].drop_duplicates().shape

(38906, 1)

In [40]:
targets = targets.merge(trans[['client_id']].drop_duplicates(), on='client_id', how='inner')

In [44]:
trans.to_parquet('../data/x5/features_sample_s.parquet')
targets.to_parquet('../data/x5/targets_sample_s.parquet')

In [3]:
target = pd.read_csv('../data/x5/clients.csv')
prods = pd.read_csv('../data/x5/products.csv')

In [4]:
data = pd.read_csv('../data/x5/purchases.csv')

In [5]:
features = ['level_3', 'level_4', 'segment_id', 'trn_sum_from_iss', 'netto', 'regular_points_received']

In [34]:
prods.head(2)

,product_id,level_1,level_2,level_3,level_4,segment_id,brand_id,vendor_id,netto,is_own_trademark,is_alcohol
0,0003020d3c,c3d3a8e8c6,c2a3ea8d5e,b7cda0ec0c,6376f2a852,123.0,394a54a7c1,9eaff48661,0.40,0,0
1,0003870676,e344ab2e71,52f13dac0c,d3cfe81323,6dc544533f,105.0,acd3dd483f,10486c3cf0,0.68,0,0


In [6]:
data = data.merge(prods, on='product_id', how='left')

In [7]:
data = data[['client_id', 'transaction_datetime'] + features]

In [8]:
data.to_parquet('../data/x5/features_raw.parquet')

In [9]:
target = pd.read_csv('../data/x5/clients.csv')
target = target[(target.age > 1) & (target.age < 99)][['client_id', 'age']]
target.age = np.where(target.age > 75, 3, np.where(target.age > 50, 2, np.where(target.age > 25, 1, 0)))

data = pd.read_parquet('../data/x5/features_raw.parquet')

In [10]:
data = data[~data.netto.isnull()]
data = data.fillna(404)

In [4]:
data['date'] = list(map(lambda x: x.split(' ')[0], data.transaction_datetime))
data['time'] = list(map(lambda x: x.split(' ')[1], data.transaction_datetime))
data = data.drop(columns=['transaction_datetime'])

In [5]:
data = data.merge(target, on='client_id', how='inner')

In [6]:
data = data.merge(pd.DataFrame({'client_id' : data['client_id'].unique(), 'client_id_n' : np.arange(data['client_id'].unique().shape[0])}), 
               on='client_id', how='inner').drop(columns=['client_id']).rename(columns={'client_id_n' : 'client_id'})

In [7]:
data[['client_id', 'date', 'time'] + ['level_3', 'level_4', 'segment_id', 'trn_sum_from_iss', 'netto', 'regular_points_received'] + ['age']].to_parquet('../data/x5/features.parquet')

In [4]:
data = pd.read_parquet('../data/x5/features.parquet')

In [7]:
data.drop(columns=['age']).to_parquet('../data/x5/features.parquet')

In [2]:
data = pd.read_parquet('../data/x5/features.parquet')

In [3]:
time_part = pd.to_datetime(data.time, format='%H:%M:%S').values.astype('int64') // 1e9

In [7]:
time_part = time_part % (24 * 60 * 60) / (24 * 60 * 60)

In [8]:
time_part

array([0.30052083, 0.30052083, 0.30052083, ..., 0.74099537, 0.74099537,
       0.74099537])

In [4]:
day_part = pd.to_datetime(data.date, format='%Y-%m-%d').values.astype('int64') // 1e9

In [9]:
day_part = day_part - min(day_part)

In [12]:
day_part = day_part // (60 * 60 * 24)

In [13]:
data['ordercol'] = day_part + time_part

In [15]:
data.to_parquet('../data/x5/features.parquet')

In [18]:
data = pd.read_parquet('../data/x5/features.parquet')
target = pd.read_parquet('../data/x5/targets.parquet')

In [19]:
sample = pd.DataFrame(data.client_id.drop_duplicates().sample(130000))
data = data.merge(sample, on='client_id', how='inner')
target = target.merge(sample, on='client_id', how='inner')

In [21]:
data.to_parquet('../data/x5/features_sample.parquet')
target.to_parquet('../data/x5/targets_sample.parquet')

In [35]:
with open(f"../coles_prep_datasets/quant_num_emb_emb16_dist_common_emb_cat_dataset_x5.pkl", "rb") as fl:
            train, val, test = pickle.load(fl)

In [36]:
train_ids = [i['client_id'] for i in train]
val_ids = [i['client_id'] for i in val]
test_ids = [i['client_id'] for i in test]

In [37]:
len(test_ids), len(val_ids), len(train_ids)

(13000, 5850, 111150)

In [40]:
set(train_ids) & set(val_ids)

set()

In [69]:
fraud = pd.read_parquet('../data/fraud/fraud_data.parquet')

In [71]:
fraud.groupby(['customer_id', 'is_fraud'])['amount'].count()

customer_id  is_fraud
0            0            961
             1            240
1            0           1130
             1            282
2            0           1377
                         ... 
4866         1            218
4867         0           1028
             1            257
4868         0           1138
             1            284
Name: amount, Length: 9738, dtype: int64

In [73]:
fraud.customer_id.unique().shape

(4869,)

In [70]:
fraud

,timestamp,amount,transaction_id,customer_id,merchant_category,merchant_type,currency,country,city_size,card_type,card_present,channel,high_risk_merchant,is_fraud
0,2024-09-30 00:00:01.034820+00:00,294.87,0,0,0,0,0,0,0,0,0,0,0,0
1,2024-09-30 00:00:01.764464+00:00,3368.97,1,1,1,1,1,1,0,0,0,1,1,1
2,2024-09-30 00:00:02.273762+00:00,102582.38,2,2,2,2,2,2,0,0,0,1,0,0
3,2024-09-30 00:00:02.297466+00:00,630.60,3,3,3,3,3,3,0,1,0,0,0,0
4,2024-09-30 00:00:02.544063+00:00,724949.27,4,4,4,4,4,4,0,2,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7483761,2024-10-30 23:59:58.926575+00:00,887.32,7477301,4278,7,5,9,10,0,1,0,1,0,0
7483762,2024-10-30 23:59:58.950801+00:00,356.06,7477302,383,2,2,5,8,0,0,0,1,0,0
7483763,2024-10-30 23:59:58.972155+00:00,391.96,7477303,4093,2,2,9,10,0,1,0,1,0,0
7483764,2024-10-30 23:59:58.996608+00:00,601.71,7477304,3133,7,5,10,11,1,4,0,1,0,0


In [2]:
a = pd.read_csv('../data/gender/transactions_d.csv').drop(columns=['term_id'])
a.head(2)

,customer_id,tr_datetime,mcc_code,tr_type,amount,description
0,39026145,0 10:23:26,4814,1030,-2245.92,"Звонки с использованием телефонов, считывающих..."
1,39026145,1 10:19:29,6011,7010,56147.89,Финансовые институты — снятие наличности автом...


In [10]:
a.drop(columns=['Unnamed: 0.1', 'Unnamed: 0']).to_csv('../data/gender/transactions_d.csv', index_label=False)

In [56]:
from sklearn.decomposition import PCA

from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer, util


class TextPreprocessor:
    def __init__(self, 
                 method, 
                 feature_names, 
                 enable_pca,  
                 compressed_dim=-1, 
                 saves_directory=None
                 ):
        self.compressed_dim = compressed_dim
        self.feature_names = feature_names
        self.enable_pca = enable_pca
        self.method = method
        if self.method == 'rubert_output':
            self.model = SentenceTransformer('sergeyzh/rubert-tiny-turbo')
            self.embedding_dim = 312
        elif self.method == 'avg_pooling_w2v':
            self.model = KeyedVectors.load_word2vec_format(hf_hub_download(repo_id="Word2vec/wikipedia2vec_ruwiki_20180420_300d", filename="ruwiki_20180420_300d.txt"))
            self.embedding_dim = 300
        elif self.method == 'avg_pooling_fasttext':
            self.model = fasttext.load_model(hf_hub_download(repo_id="facebook/fasttext-ru-vectors", filename="model.bin")) 
            self.embedding_dim = 300
        else:
            raise Exception('No method of text feature encoding with such name exists')

        self.pca_models = dict()
        if self.enable_pca:
            for fe in self.feature_names:
                self.pca_models[fe] =  PCA(n_components=self.compressed_dim)

        self.unk_str = None

    def calc_embeddings(self, X, idcol_name, fit_pca=True):
        ids = []
        for sample in X:
            ids.append(sample[idcol_name])
        
        embeds = dict()
        for fe in self.feature_names:
            def vclear(s):
                s = " ".join(s[fe])
                return re.sub(r"[^а-яА-Я0-9\s]+|\s{2,}", " ", s.lower()).strip().split(" ")

            if self.method == 'rubert_output':

                def get_emb(sample):
                    return self.model.encode(sample[fe])

                embeds[fe] = list(map(get_emb, X))
            elif self.method == 'avg_pooling_w2v':
                if self.unk_str is None:
                    vocabulary = list(set(np.concatenate(list(map(vclear, X)))) - {"", " "})
                    self.unk_str = self.unk_detection(vocabulary)

                def sclear(s):
                    sts = []
                    for st in s[fe]:
                        st = re.sub(r"[^а-яА-Я0-9\s]+|\s{2,}", " ", st.lower())
                        if self.unk_str != '':
                            st = re.sub(rf"{self.unk_str}", "unk", st)
                        sts.append(np.mean(self.model[*re.sub(r"\s+", " ", st).strip().split(" ")], axis=0))
                    return sts

                embeds[fe] = list(map(sclear, X))

            elif self.method == 'avg_pooling_fasttext':
                if self.unk_str is None:
                    vocabulary = list(set(np.concatenate(list(map(vclear, X)))) - {"", " "})
                    self.unk_str = self.unk_detection(vocabulary)

                def sclear(s):
                    sts = []
                    for st in s[fe]:
                        st = re.sub(r"[^а-яА-Я0-9\s]+|\s{2,}", " ", st.lower())
                        if self.unk_str != '':
                            st = re.sub(rf"{self.unk_str}", "unk", st)
                        sts.append(self.model.get_sentence_vector(re.sub(r"\s+", " ", st).strip()))
                    return sts

                embeds[fe] = list(map(sclear, X))

            if self.enable_pca:
                if fit_pca:
                    self.pca_models[fe].fit(np.concatenate(embeds[fe]))
                embeds[fe] = list(map(self.pca_models[fe].transform, embeds[fe]))

        return pd.DataFrame({idcol_name : ids} | embeds)

    def unk_detection(self, voc):
        unk_str = ""
        for w in voc:
            try:
                self.model[w]
            except:
                unk_str = unk_str + w + "|"
        return unk_str[:-1]

In [4]:
from sklearn.decomposition import PCA

from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer, util


class TextPreprocessor:
    def __init__(self, 
                 method, 
                 feature_names, 
                 enable_pca,  
                 compressed_dim=-1, 
                 saves_directory=None
                 ):
        self.compressed_dim = compressed_dim
        self.feature_names = feature_names
        self.enable_pca = enable_pca
        self.method = method
        if self.method == 'rubert_output':
            self.model = SentenceTransformer('sergeyzh/rubert-tiny-turbo')
            self.embedding_dim = 312
        elif self.method == 'avg_pooling_w2v':
            self.model = KeyedVectors.load_word2vec_format(hf_hub_download(repo_id="Word2vec/wikipedia2vec_ruwiki_20180420_300d", filename="ruwiki_20180420_300d.txt"))
            self.embedding_dim = 300
        elif self.method == 'avg_pooling_fasttext':
            self.model = fasttext.load_model(hf_hub_download(repo_id="facebook/fasttext-ru-vectors", filename="model.bin")) 
            self.embedding_dim = 300
        else:
            raise Exception('No method of text feature encoding with such name exists')

        self.pca_models = dict()
        if self.enable_pca:
            for fe in self.feature_names:
                self.pca_models[fe] =  PCA(n_components=self.compressed_dim)

        self.unk_str = dict()

    def calc_embeddings(self, X, fit_pca=True):     
        for fe in self.feature_names:
            embeds = X[fe].values
            def vclear(s):
                s = " ".join(s)
                return re.sub(r"[^а-яА-Я0-9\s]+|\s{2,}", " ", s.lower()).strip().split(" ")

            if self.method == 'rubert_output':
                batch_size = 10000
                embeds_list = []

                for i in tqdm(range(0, (X.shape[0] // batch_size) + 1)):
                    if i * batch_size == X.shape[0]:
                        break
                    embeds_list.append(self.model.encode(embeds[i * batch_size : min((i + 1) * batch_size, X.shape[0])]))

                embeds = np.concatenate(embeds_list)
            elif self.method == 'avg_pooling_w2v':
                if fe not in self.unk_str:
                    vocabulary = list(set(vclear(embeds)) - {"", " "})
                    self.unk_str[fe] = self.unk_detection(vocabulary)

                def sclear(s):
                    s = re.sub(r"[^а-яА-Я0-9\s]+|\s{2,}", " ", s.lower())
                    if self.unk_str[fe] != '':
                        s = re.sub(rf"{self.unk_str[fe]}", "unk", s)
                    return np.mean(self.model[*re.sub(r"\s+", " ", s).strip().split(" ")], axis=0)

                embeds = np.array(list(map(sclear, embeds)))

            elif self.method == 'avg_pooling_fasttext':
                if fe not in self.unk_str:
                    vocabulary = list(set(vclear(embeds)) - {"", " "})
                    self.unk_str[fe] = self.unk_detection(vocabulary)

                def sclear(s):
                    s = re.sub(r"[^а-яА-Я0-9\s]+|\s{2,}", " ", s.lower())
                    if self.unk_str[fe] != '':
                        s = re.sub(rf"{self.unk_str[fe]}", "unk", s)
                    return self.model.get_sentence_vector(re.sub(r"\s+", " ", s).strip())

                embeds = np.array(list(map(sclear, embeds)))

            if self.enable_pca:
                if fit_pca:
                    self.pca_models[fe].fit(embeds)
                X[fe] = list(self.pca_models[fe].transform(embeds))

        return X

    def unk_detection(self, voc):
        unk_str = ""
        for w in voc:
            try:
                self.model[w]
            except:
                unk_str = unk_str + w + "|"
        return unk_str[:-1]

In [1]:
from ptls.preprocessing.text_preprocessing import TextPreprocessor

In [10]:
from sklearn.model_selection import train_test_split
with open(f"../coles_prep_datasets/baseline_with_text_dataset_gender.pkl", "rb") as fl:
            dataset = pickle.load(fl)

train, test = train_test_split(dataset, test_size=0.1, random_state=42)

In [37]:
a = torch.tensor(np.arange(10).reshape(2, 5))
torch.cat([a, a, a], dim=0)

tensor([[0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9]], dtype=torch.int32)

In [46]:
from torch import nn as nn
emb = nn.Embedding(num_embeddings=10, embedding_dim=10)

In [66]:
report = pd.read_csv('../coles_experiments/airi_conf_my_mods_48_gender.csv')
report.value = np.round(report.value, 4)
report.value += 0.01

In [67]:
report[report.dataset == 'test'].drop(columns='dataset')

,Unnamed: 0,exp_name,metric,value
0,0,quant_100_emb48_dist_common_emb_sum,recall_top_k,0.8621
1,1,quant_100_emb48_dist_common_emb_sum,accuracy_lgbm_boosting,0.7826
2,2,quant_100_emb48_dist_common_emb_sum,roc_auc_lgbm_boosting,0.8754
5,5,quant_100_emb48_dist_common_emb_cat,recall_top_k,0.8689
6,6,quant_100_emb48_dist_common_emb_cat,accuracy_lgbm_boosting,0.7779
7,7,quant_100_emb48_dist_common_emb_cat,roc_auc_lgbm_boosting,0.8742
10,10,deeptlf9_emb48_disc_common_emb_mean,recall_top_k,0.7839
11,11,deeptlf9_emb48_disc_common_emb_mean,accuracy_lgbm_boosting,0.7849
12,12,deeptlf9_emb48_disc_common_emb_mean,roc_auc_lgbm_boosting,0.8780
15,15,deeptlf9_emb48_disc_common_emb_cat,recall_top_k,0.7901


In [40]:
emb(torch.tensor([[1, 4], [2, 5]])).shape

torch.Size([2, 2, 3])

In [5]:
text_prep = TextPreprocessor('rubert_output', ['description'], enable_pca=True, compressed_dim=48)

In [6]:
#bert 1043/100
#v2w 266/19
#ft 285/20

tt = time.time()
a = text_prep.calc_embeddings(a, fit_pca=True)
time.time() - tt

  0%|          | 0/685 [00:00<?, ?it/s]

988.1768774986267

In [7]:
a.description[0].shape

(48,)

In [61]:
tt = time.time()
embeds_test = text_prep.calc_embeddings(test, 'client_id', fit_pca=False)
time.time() - tt

20.72563409805298

In [54]:
{1:2} | {3:4, 5:6}

{1: 2, 3: 4, 5: 6}

In [33]:
pd.DataFrame({'client_id' : ids, 'description_emb' : embeds}).head(3)

,client_id,description_emb
0,78833879,"[[0.38614854, 0.0028899945, 0.006633103, 0.013..."
1,6710884,"[[-0.25542307, -0.24894574, -0.20159115, -0.00..."
2,11032699,"[[-0.06324831, 0.41056132, -0.20175041, -0.017..."


In [16]:
model = SentenceTransformer('sergeyzh/rubert-tiny-turbo')
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 2048, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 312, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [25]:
import torch
a = pd.DataFrame([])
a['d'] = [torch.tensor([1, 2, 3, 4]), torch.tensor([1, 2, 3, 4])]

In [33]:
tuple(a.iloc[:, :].values)

(array([tensor([1, 2, 3, 4])], dtype=object),
 array([tensor([1, 2, 3, 4])], dtype=object))

In [18]:
ids = []
for el in tqdm(dataset):
    ids.append(el['client_id'])

  0%|          | 0/15000 [00:00<?, ?it/s]

In [28]:
np.stack([np.array(ids).reshape(-1, 1), embeds])

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (15000,) + inhomogeneous part.

In [14]:
FE = 'description'

def get_emb(sample):
    return model.encode(sample[FE])

embeds = list(map(get_emb, dataset))

In [16]:
embeds[0]

array([[ 0.02567611,  0.01060615,  0.01709403, ...,  0.02607744,
         0.04172674, -0.03507769],
       [ 0.02567614,  0.01060615,  0.01709399, ...,  0.02607744,
         0.04172675, -0.03507769],
       [ 0.01457031, -0.0086526 , -0.00497847, ..., -0.02990391,
         0.07487129, -0.02889231],
       ...,
       [ 0.02567614,  0.01060615,  0.01709399, ...,  0.02607744,
         0.04172675, -0.03507769],
       [ 0.02567611,  0.01060615,  0.01709403, ...,  0.02607744,
         0.04172674, -0.03507769],
       [ 0.02567614,  0.01060615,  0.01709399, ...,  0.02607744,
         0.04172675, -0.03507769]], dtype=float32)

In [6]:
n_batches = 600

bs = a.shape[0] // n_batches + 1
print(bs)
embeds = []

embeds.append(model.encode(a.description.values[bs * i : min(bs * (i + 1), a.shape[0])]))

11416


  0%|          | 0/600 [00:00<?, ?it/s]


KeyboardInterrupt



In [50]:
model = KeyedVectors.load_word2vec_format(hf_hub_download(repo_id="Word2vec/nlpl_183", filename="model.bin"), binary=True, unicode_errors="ignore")

In [4]:
model = KeyedVectors.load_word2vec_format(hf_hub_download(repo_id="Word2vec/wikipedia2vec_ruwiki_20180420_300d", filename="ruwiki_20180420_300d.txt"))

In [13]:
model['unk']

array([ 0.1212, -0.0998,  0.3103,  0.0883,  0.0277,  0.1799, -0.0321,
       -0.211 , -0.136 , -0.0872, -0.0071, -0.2451, -0.0691, -0.0337,
       -0.0653, -0.1571,  0.0352, -0.2335,  0.0252, -0.1229, -0.0031,
        0.0832, -0.0563,  0.1999,  0.017 ,  0.0409,  0.0217, -0.04  ,
        0.1983, -0.1369,  0.321 , -0.0191, -0.0337,  0.1558, -0.1772,
        0.2579, -0.1205, -0.1987,  0.1356, -0.0007,  0.386 ,  0.0387,
        0.1059, -0.2074,  0.1212, -0.0807,  0.4344, -0.0963,  0.01  ,
       -0.0349,  0.0712,  0.0015,  0.1355, -0.2075,  0.0589, -0.1717,
       -0.1524, -0.121 , -0.0405,  0.3274, -0.2109,  0.0611,  0.2127,
        0.2151,  0.2507,  0.0286, -0.0351,  0.0698,  0.26  , -0.0205,
       -0.2078, -0.0145,  0.1403, -0.0604, -0.1618,  0.0648, -0.2767,
        0.0238, -0.013 ,  0.1057, -0.24  , -0.0437, -0.1046, -0.166 ,
        0.0219, -0.2094,  0.0384, -0.122 , -0.0578,  0.1685, -0.1476,
       -0.0752,  0.0128,  0.3137,  0.0152,  0.0401,  0.115 ,  0.1683,
       -0.1233,  0.0

In [31]:
def unk_detection(voc):
    unk_str = ""
    for w in voc:
        try:
            model[w]
        except:
            unk_str = unk_str + w + "|"
    return unk_str[:-1]

In [39]:
def vclear(s):
    s = " ".join(s['description'])
    return re.sub(r"[^а-яА-Я0-9\s]+|\s{2,}", " ", s.lower()).strip().split(" ")

vocabulary = list(set(np.concatenate(list(map(vclear, dataset)))) - {"", " "})
unk_str = unk_detection(vocabulary)

UNK_STR = unk_str

In [40]:
unk_str

'микрофильмирующее|фотокопировальное|ортодонтисты|фотоприборов|автогрузоперевозки|попереезду|билльярд|лабороторное|дальные|беспошлинные|салоты|разжиженный|комиссионки|\xa0с|превыш|обивочный|овердрафте|веломагазины'

In [46]:
POOLING_FUNC = lambda x: np.mean(x, axis=0)

def sclear(s):
    sts = []
    for st in s['description']:
        st = re.sub(r"[^а-яА-Я0-9\s]+|\s{2,}", " ", st.lower())
        if UNK_STR != '':
            st = re.sub(rf"{UNK_STR}", "unk", st)
        sts.append(POOLING_FUNC(model[*re.sub(r"\s+", " ", st).strip().split(" ")]))
    return sts

In [47]:
tt = time.time()
embeds = list(map(sclear, dataset))
time.time() - tt

221.76157331466675

In [162]:
a.description.shape

(6849346,)

In [32]:
with open('../data/gender/description_emb_w2v.pkl', 'wb') as fl:
    pickle.dump(embeds, fl)

In [52]:
model.get_sentence_vector('привет пока')

AttributeError: 'KeyedVectors' object has no attribute 'get_sentence_vector'

In [8]:
from icu_tokenizer import Tokenizer
tokenizer = Tokenizer(lang='ru')

tokenizer.tokenize('Мама мыла раму/дверь')

['Мама', 'мыла', 'раму', '/', 'дверь']

In [13]:
tokenizer.tokenize(" ".join(a.description.values))[:1000]

['Звонки',
 'с',
 'использованием',
 'телефонов',
 ',',
 'считывающих',
 'магнитную',
 'ленту',
 'Оплата',
 'услуги',
 '.',
 'Банкоматы',
 'СБ',
 'РФ',
 'Финансовые',
 'институты',
 '—',
 'снятие',
 'наличности',
 'автоматически',
 'Взнос',
 'наличных',
 'через',
 'АТМ',
 '(',
 'в',
 'своем',
 'тер.банке',
 ')',
 'Денежные',
 'переводы',
 'Списание',
 'с',
 'карты',
 'по',
 'операции',
 '“',
 'перевода',
 'с',
 'карты',
 'на',
 'карту',
 '”',
 'через',
 'АТМ',
 '(',
 'в',
 'пределах',
 'одного',
 'тер.банка',
 ')',
 'Различные',
 'продовольственные',
 'магазины',
 '—',
 'рынки',
 ',',
 'магазины',
 'со',
 'спец',
 '-',
 'ассортиментом',
 ',',
 'продажа',
 'полуфабрикатов',
 ',',
 'фирменных',
 'блюд',
 ',',
 'продажа',
 'с',
 'помощью',
 'торговых',
 'автоматов',
 'Покупка',
 '.',
 'POS',
 'ТУ',
 'СБ',
 'РФ',
 'Различные',
 'продовольственные',
 'магазины',
 '—',
 'рынки',
 ',',
 'магазины',
 'со',
 'спец',
 '-',
 'ассортиментом',
 ',',
 'продажа',
 'полуфабрикатов',
 ',',
 'фирменных'

In [14]:
model = fasttext.load_model(hf_hub_download(repo_id="facebook/fasttext-ru-vectors", filename="model.bin"))

In [15]:
model.get_sentence_vector("")

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [38]:
def vclear(s):
    return re.sub(r"[^а-яА-Я0-9\s]+|\s{2,}", " ", s.lower()).strip().split(" ")

vocabulary = list(set(vclear(" ".join(a.description.values))) - {"", " "})
unk_str = unk_detection(vocabulary)

UNK_STR = unk_str

In [86]:
#fasttext pooling: avg(l2(w1) ... l2(wn)) | wi : l2(wi) > 0
#POOLING_FUNC = lambda x: np.mean(x, axis=0)

def sclear(s):
    s = re.sub(r"[^а-яА-Я0-9\s]+|\s{2,}", " ", s.lower())
    if UNK_STR != '':
        s = re.sub(rf"{UNK_STR}", "unk", s)
    return model.get_sentence_vector(re.sub(r"\s+", " ", s).strip())

In [87]:
tt = time.time()
embeds = a.description.apply(sclear)
time.time() - tt

207.3992931842804

In [75]:
model.get_word_vector('привет') - model.get_sentence_vector('привет')

array([-6.15190715e-03,  1.45997386e-03,  6.65750355e-03,  3.42515483e-03,
       -1.40691083e-03,  3.34920362e-03, -2.58298032e-03, -3.97892669e-03,
        3.16701480e-05, -3.39646125e-04, -6.67248666e-03, -5.93264028e-03,
       -4.92756814e-03, -3.66312265e-03,  2.28953920e-03,  3.78040597e-03,
        1.81317679e-04,  1.06833056e-02,  3.89794260e-03, -8.88168067e-03,
        7.08658993e-03,  4.98021767e-03, -1.92857906e-03, -4.62485105e-03,
       -2.11540051e-03, -8.18870030e-04,  2.92108767e-03, -4.52553481e-03,
       -4.18835878e-03, -3.68727371e-03, -3.29165906e-03, -5.33176586e-03,
       -1.44692603e-03, -1.34378672e-02, -3.21744010e-03,  2.79212371e-03,
        9.85278189e-03,  8.92252475e-03, -2.95028277e-03,  1.05811805e-02,
        8.59758258e-03, -5.59357926e-03,  1.66582689e-03, -1.15350471e-04,
        6.83879107e-03, -9.85962152e-03,  7.71176815e-03,  6.14222139e-04,
       -4.21479344e-03,  5.47670946e-03,  1.71641074e-03, -3.76369059e-03,
       -4.56809253e-03,  

In [83]:
def l2_norm(x):
   return np.sqrt(np.sum(x**2))

def get_normed(w):
    emb = model.get_word_vector(w)
    l2 = l2_norm(emb)
    return emb / l2

l2_norm(model.get_word_vector('стол'))

1.4886659

In [84]:
np.mean(np.array([get_normed('привет'), get_normed('мама'), get_normed('стол')]), axis=0) - model.get_sentence_vector('привет мама стол')

array([ 0.0000000e+00,  3.7252903e-09,  3.7252903e-09, -1.8626451e-09,
        3.7252903e-09, -8.3819032e-09,  3.7252903e-09,  3.7252903e-09,
        3.7252903e-09,  0.0000000e+00,  0.0000000e+00, -7.4505806e-09,
        5.5879354e-09,  9.3132257e-10, -7.4505806e-09,  0.0000000e+00,
        4.6566129e-10, -3.7252903e-09,  0.0000000e+00,  7.4505806e-09,
       -1.8626451e-09,  0.0000000e+00,  0.0000000e+00,  3.7252903e-09,
        0.0000000e+00,  5.5879354e-09, -5.5879354e-09,  0.0000000e+00,
       -3.7252903e-09,  0.0000000e+00, -1.8626451e-09,  1.8626451e-09,
       -3.7252903e-09,  3.7252903e-09,  0.0000000e+00,  2.3283064e-10,
       -3.7252903e-09,  7.4505806e-09,  7.4505806e-09, -7.4505806e-09,
        0.0000000e+00,  5.5879354e-09, -4.6566129e-09, -1.3969839e-09,
        3.7252903e-09,  1.0244548e-08, -3.7252903e-09,  3.7252903e-09,
        0.0000000e+00, -1.4901161e-08,  1.8626451e-09,  0.0000000e+00,
        3.7252903e-09, -1.3038516e-08,  0.0000000e+00,  0.0000000e+00,
      

In [65]:
with open('../data/gender/description_emb_fasttext.pkl', 'wb') as fl:
    pickle.dump(embeds, fl)

## PCA

In [39]:
with open('../data/gender/description_emb.pkl', 'wb') as fl:
    pickle.dump(embeds, fl)

In [19]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100)
pca.fit(text_prep.unk_str)

PCA(n_components=100)

In [43]:
pca.explained_variance_ratio_

array([4.11983848e-01, 1.82683215e-01, 1.17359117e-01, 4.15447764e-02,
       3.65382247e-02, 2.58944314e-02, 1.96549576e-02, 1.84964575e-02,
       1.35569340e-02, 1.27284545e-02, 1.03534264e-02, 9.42146592e-03,
       8.53461586e-03, 7.71311671e-03, 6.33986108e-03, 6.18652347e-03,
       5.27257798e-03, 4.80531575e-03, 4.02716547e-03, 3.78971128e-03,
       3.45425378e-03, 3.18450201e-03, 2.62481021e-03, 2.53438717e-03,
       2.41325563e-03, 2.17695837e-03, 2.08504405e-03, 1.99661171e-03,
       1.70449843e-03, 1.68998551e-03, 1.55007816e-03, 1.43566006e-03,
       1.30059570e-03, 1.23281288e-03, 1.18375837e-03, 1.12457294e-03,
       1.08468160e-03, 9.97078838e-04, 9.95499548e-04, 8.47157440e-04,
       8.31813377e-04, 7.47600337e-04, 7.10582652e-04, 7.00251316e-04,
       6.73219678e-04, 6.44059386e-04, 5.91815449e-04, 5.79622632e-04,
       5.73611818e-04, 5.02045907e-04, 4.74327651e-04, 4.57427785e-04,
       4.15779272e-04, 4.12594032e-04, 4.01723431e-04, 3.82267259e-04,
      

In [20]:
a['description'] = list(pca.transform(np.array(text_prep.unk_str)))

In [21]:
a.head()

,customer_id,tr_datetime,mcc_code,tr_type,amount,description
0,39026145,0 10:23:26,4814,1030,-2245.92,"[-0.045378677052174625, 0.183916945841218, 0.1..."
1,39026145,1 10:19:29,6011,7010,56147.89,"[-0.1646443309647025, -0.13257749688608203, 0...."
2,39026145,1 10:20:56,4829,2330,-56147.89,"[-0.16519283242089824, 0.05542501456793926, -0..."
3,39026145,1 10:39:54,5499,1010,-1392.47,"[0.12523907708286278, 0.03020962233991474, 0.0..."
4,39026145,2 15:33:42,5499,1010,-920.83,"[0.12523907708286278, 0.03020962233991474, 0.0..."


In [3]:
from datetime import timedelta
import random

a['time'] = [i[-8:] for i in a.tr_datetime]

padded_time = a['tr_datetime'].str.pad(15, 'left', '0')
day_part = padded_time.str[:6].astype(float)
time_part = pd.to_datetime(padded_time.str[7:], format='%H:%M:%S').values.astype('int64') // 1e9
time_part = time_part % (24 * 60 * 60) / (24 * 60 * 60)

a.tr_datetime = day_part + time_part

a['date'] = [str(i + datetime.timedelta(days=random.randint(40000, 40600)))[:10] for i in pd.to_datetime(a.time, format='%H:%M:%S')]

In [34]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

class TimePreprocessor:
    def __init__(self, idcol, ordercol, datecol=None, timecol=None, mode='all', exclude_list=[]):
        """
        ordercol: must contain positive integers
        datecol: must contain transaction date in format of string YYYY-MM-DD
        timecol: must contain transaction time in format of string HH:MM:SS
        mode: 'all' - collect all features, 'cat' - collect categorical features, 'num' - collect numeric features
        """
        self.idcol = idcol
        self.ordercol = ordercol
        self.datecol = datecol
        self.timecol = timecol
        self.mode = mode

        self.cat_features = []
        self.num_features = []
        self.exclude_list = exclude_list

        self.scalers = dict()

    def time_to_seconds(self, t):
        h, m, s = map(int, t.split(':'))
        return h * 3600 + m * 60 + s

    def normalize(self, f):
        return  (X[self.ordercol] - X[self.ordercol].mean())/(X[self.ordercol].max() - X[self.ordercol].min())

    def fit(self, X):
        if self.ordercol is not None:
            ordercol_deltas = X.groupby(self.idcol)[self.ordercol].transform(lambda x: x.sort_values().diff()).fillna(1)
            first_values = X.groupby(self.idcol)[self.ordercol].agg('min')
            if (ordercol_deltas != 1.0).sum() != 0 or  (first_values == first_values.iloc[0]).sum() != 0:
                self.num_features.append('TIME_ordercol_num')
                scaler = MinMaxScaler()
                scaler.fit(X[self.ordercol].values.reshape(-1, 1))
                self.scalers['TIME_ordercol_num'] = scaler
                if X[self.ordercol].unique().shape[0] < 1000:
                    self.cat_features.append('TIME_ordercol_cat')

        if self.timecol is not None:
            X['time_seconds'] = X[self.timecol].apply(self.time_to_seconds)
            if self.datecol is not None:
                time_deltas = X.sort_values(by=[self.idcol, self.datecol, 'time_seconds']).groupby([self.idcol, self.datecol])['time_seconds'].diff().fillna(3601)
                if (time_deltas < 3600).sum() > 0.1 * time_deltas.shape[0]:
                    self.num_features.append('TIME_daily_seconds_sin')
                    self.num_features.append('TIME_daily_seconds_cos')
            else:
                self.num_features.append('TIME_time_seconds')
                    
            self.cat_features.append('TIME_hour')
            self.num_features.append('TIME_hour_sin')
            self.num_features.append('TIME_hour_cos')
        if self.datecol is not None:
            dates = pd.to_datetime(X[self.datecol], format='%Y-%m-%d')
            date_range = (max(dates) - min(dates)).days
            if date_range > 366:
                self.cat_features.append('TIME_month')
            if date_range > 90:  
                self.cat_features.append('TIME_monthday')
                self.num_features.append('TIME_monthday_sin')
                self.num_features.append('TIME_monthday_cos')
            if date_range > 30:
                self.cat_features.append('TIME_weekday')
                self.num_features.append('TIME_weekday_sin')
                self.num_features.append('TIME_weekday_cos')
        self.cat_features = sorted(list(set(self.cat_features) - set(self.exclude_list)))
        self.num_features = sorted(list(set(self.num_features) - set(self.exclude_list)))


    def transform(self, X):
        if self.datecol is not None:
            if self.timecol is not None:
                timestamp = pd.to_datetime(X[self.datecol] + " " + X[self.timecol], format='%Y-%m-%d %H:%M:%S')
            else:
                timestamp = pd.to_datetime(X[self.datecol], format='%Y-%m-%d')
        elif self.timecol is not None:
            timestamp = pd.to_datetime(X[self.timecol], format='%H:%M:%S')

        if len({'TIME_daily_seconds_sin', 'TIME_daily_seconds_cos'} & set(self.num_features)) > 0:
            time_seconds = X[self.timecol].apply(self.time_to_seconds)

        if self.mode == 'cat' or self.mode == 'all':
            if 'TIME_ordercol_cat' in self.cat_features:
                X['TIME_ordercol_cat'] = X[self.ordercol] 

            if 'TIME_hour' in self.cat_features:
                X['TIME_hour'] =  timestamp.dt.hour

            if 'TIME_month' in self.cat_features:
                X['TIME_month'] =  timestamp.dt.month

            if 'TIME_monthday' in self.cat_features:
                X['TIME_monthday'] =  timestamp.dt.day

            if 'TIME_weekday' in self.cat_features:
                X['TIME_weekday'] =  timestamp.dt.weekday
            
        if self.mode == 'num' or self.mode == 'all':
            if 'TIME_ordercol_num' in self.num_features:
                X['TIME_ordercol_num'] =  self.scalers['TIME_ordercol_num'].transform(X[self.ordercol].values.reshape(-1, 1))

            if 'TIME_daily_seconds_sin' in self.num_features:
                X['TIME_daily_seconds_sin'] =  np.sin(2 * np.pi * time_seconds / (60 * 60 * 24))

            if 'TIME_daily_seconds_cos' in self.num_features:
                X['TIME_daily_seconds_cos'] =  np.cos(2 * np.pi * time_seconds / (60 * 60 * 24))

            if 'TIME_hour_sin' in self.num_features:
                X['TIME_hour_sin'] =  np.sin(2 * np.pi * timestamp.dt.hour.values / 24)

            if 'TIME_hour_cos' in self.num_features:
                X['TIME_hour_cos'] =  np.cos(2 * np.pi * timestamp.dt.hour.values / 24)

            if 'TIME_monthday_sin' in self.num_features:
                X['TIME_monthday_sin'] =  np.sin(2 * np.pi * timestamp.dt.day.values / 30)

            if 'TIME_monthday_cos' in self.num_features:
                X['TIME_monthday_cos'] =  np.cos(2 * np.pi * timestamp.dt.day.values / 30)

            if 'TIME_weekday_sin' in self.num_features:
                X['TIME_weekday_sin'] =  np.sin(2 * np.pi * timestamp.dt.weekday.values / 7)

            if 'TIME_weekday_cos' in self.num_features:
                X['TIME_weekday_cos'] =  np.cos(2 * np.pi * timestamp.dt.weekday.values / 7)

        return X

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

In [4]:
from ptls.preprocessing.time_preprocessing import TimePreprocessor

tp = TimePreprocessor(idcol='customer_id', 
                      ordercol='tr_datetime',
                      datecol='date', 
                      timecol='time',
                      mode='all')

In [5]:
tp.fit(a)

C:\Users\toppc\Documents\diploma\ptls-glove\ptls\preprocessing\time_preprocessing\preprocessor.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['time_seconds'] = X[self.timecol].apply(self.time_to_seconds)


In [6]:
a = tp.transform(a)

In [7]:
a

,Unnamed: 0,customer_id,tr_datetime,mcc_code,tr_type,amount,term_id,time,date,TIME_hour,...,TIME_weekday,TIME_ordercol_num,TIME_daily_seconds_sin,TIME_daily_seconds_cos,TIME_hour_sin,TIME_hour_cos,TIME_monthday_sin,TIME_monthday_cos,TIME_weekday_sin,TIME_weekday_cos
0,0,39026145,0.432940,4814,1030,-2245.92,NaN,10:23:26,2009-12-26,10,...,5,0.000947,0.408994,-0.912537,0.500000,-8.660254e-01,-7.431448e-01,0.669131,-0.974928,-0.222521
1,1,39026145,1.430197,6011,7010,56147.89,NaN,10:19:29,2010-11-04,10,...,3,0.003130,0.424660,-0.905353,0.500000,-8.660254e-01,7.431448e-01,0.669131,0.433884,-0.900969
2,2,39026145,1.431204,4829,2330,-56147.89,NaN,10:20:56,2009-08-18,10,...,1,0.003132,0.418924,-0.908021,0.500000,-8.660254e-01,-5.877853e-01,-0.809017,0.781831,0.623490
3,3,39026145,1.444375,5499,1010,-1392.47,NaN,10:39:54,2009-11-01,10,...,6,0.003161,0.342430,-0.939543,0.500000,-8.660254e-01,2.079117e-01,0.978148,-0.781831,0.623490
4,4,39026145,2.648403,5499,1010,-920.83,NaN,15:33:42,2010-05-31,15,...,0,0.005795,-0.803078,-0.595875,-0.707107,-7.071068e-01,2.079117e-01,0.978148,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6849341,6849341,61870738,453.668773,5499,1010,-5176.84,10217113,16:03:02,2009-11-19,16,...,3,0.992711,-0.872567,-0.488494,-0.866025,-5.000000e-01,-7.431448e-01,-0.669131,0.433884,-0.900969
6849342,6849342,61870738,454.454861,5411,1010,-1652.77,022915,10:54:60,2009-12-15,10,...,1,0.994431,0.279829,-0.960050,0.500000,-8.660254e-01,5.665539e-16,-1.000000,0.781831,0.623490
6849343,6849343,61870738,454.599988,5499,1010,-4687.23,10217113,14:23:59,2010-11-21,14,...,6,0.994749,-0.587726,-0.809060,-0.500000,-8.660254e-01,-9.510565e-01,-0.309017,-0.781831,0.623490
6849344,6849344,61870738,454.674919,5541,1110,-4491.83,RU570124,16:11:53,2009-12-15,16,...,1,0.994912,-0.890775,-0.454444,-0.866025,-5.000000e-01,5.665539e-16,-1.000000,0.781831,0.623490


In [166]:
b = pd.to_datetime(a.time, format='%H:%M:%S')

In [249]:
from datetime import timedelta
import random

c = [str(i + datetime.timedelta(days=random.randint(40000, 40600)))[:10] for i in b]

In [250]:
a['date'] = c

In [198]:
datetime.datetime.strptime(c[1], '%Y-%M-%d') - datetime.datetime.strptime(c[31223], '%Y-%M-%d')

datetime.timedelta(days=384, seconds=85920)

In [207]:
[datetime.datetime.strptime(i, '%Y-%M-%d') for i in c]

KeyboardInterrupt: 

In [274]:
w = pd.to_datetime(a['date'] + " " + a['time'], format='%Y-%m-%d %H:%M:%S')

In [277]:
w.dt.hour

0          10
1          10
2          10
3          10
4          15
           ..
6849341    16
6849342    10
6849343    14
6849344    16
6849345    18
Length: 6849346, dtype: int32

In [210]:
d = pd.to_datetime(a['date'], format='%Y-%m-%d')

In [256]:
a['date'] + " " + a['time']

0          2009-07-11 10:23:26
1          2010-10-02 10:19:29
2          2011-01-11 10:20:56
3          2010-03-30 10:39:54
4          2010-02-05 15:33:42
                  ...         
6849341    2011-02-12 16:03:02
6849342    2010-04-18 10:54:60
6849343    2010-04-10 14:23:59
6849344    2009-08-09 16:11:53
6849345    2010-08-27 18:06:30
Length: 6849346, dtype: object

In [239]:
a['time'].apply(lambda x: int(x[:2]))

0          10
1          10
2          10
3          10
4          15
           ..
6849341    16
6849342    10
6849343    14
6849344    16
6849345    18
Name: time, Length: 6849346, dtype: int64